# 3D-Mirage:

A **self-contained** training + evaluation pipeline for the 3D-Mirage


In [ ]:
# === Dependencies + imports ===
import importlib, subprocess, sys
def _need(mod, pip=None):
    try:
        importlib.import_module(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip or mod], check=False)
for _m, _p in [("peft", "peft"), ("shapely", "shapely"),
               ("cv2", "opencv-python-headless"), ("huggingface_hub", "huggingface_hub")]:
    _need(_m, _p)

import os, math, json, time, random, hashlib, shutil
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, cv2
from PIL import Image
from shapely.geometry import Polygon
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from peft import LoraConfig, get_peft_model
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !cp -r /content/drive/MyDrive/701_StillsRaw_full.zip /content/
# !cp /content/drive/MyDrive/PennFudanPed.zip /content/

# !mkdir -p neg_data
# !unzip PennFudanPed.zip
# !mv PennFudanPed/PNGImages/* neg_data/
# !unzip -q 701_StillsRaw_full.zip -d camvid_tmp

# !find camvid_tmp -type f \( -iname '*.png' -o -iname '*.jpg' -o -iname '*.jpeg' \) -exec mv -n -t neg_data {} +
# !rm -rf camvid_tmp 701_StillsRaw_full.zip
# !echo "Copied $(ls -1 neg_data | wc -l) files into neg_data/"

In [ ]:
SEED       = 1337
MID        = "depth-anything/Depth-Anything-V2-Large-hf"   # teacher + student backbone
SZ         = (640, 480)            # (W, H) processing size
HF_REPO    = "hdnndh/3D-Mirage"    # released positives (468 scenes), loose files at repo root
ROOT       = "3dmirage"            # local dir the HF dataset is downloaded into
NEG        = "neg_data"            # optional negatives dir (CamVid + PennFudanPed); auto-detected below
TEST_RATIO = 0.10                  # deterministic held-out fraction
AUG_MULT   = 4                     # train-time augmentation replication
BS         = 8                     # batch size 
E          = 1                     # epochs (raise for real training)
NW         = 2                     # dataloader workers (Colab-safe)
LR, WD, GRAD_CLIP = 1e-4, 0.01, 1.0

W_FLAT  = 1.0   # Lf          : ROI Laplacian flatness
W_BGD   = 1.0   # LbD         : background depth preservation
W_BGL   = 1.0   # LbL         : background Laplacian preservation
W_SEAM  = 1.0   # Lseam       : seam tether to smoothed teacher (low-gradient ROI ring)
W_EDGE  = 1.0   # Ledge       : edge-subset 2nd-order match (high-gradient ROI ring)
W_EDGEG = 1.0   # LedgeG      : guard-ring 2nd-order match
W_ROI   = 1.0   # Lroi_exp    : gated plane-mixture HKR (K fitted local planes + null)
W_GCE   = 1.0   # LgateCE     : gate cross-entropy (target from plane-fit residuals)
W_GENT  = 0.0
W_ANCH  = 1.0   # Lroi_anchor : best-plane anchor
W_FULL  = 1.0   # facF        : full-image branch weight relative to the crop branch

# --- Negatives (optional; AUTO-DETECTED, no download performed here) ---
# To train WITH negatives, drop the two public datasets' frames under  neg_data/  (on Colab:
# /content/neg_data/ ) BEFORE running -- e.g.:
#     neg_data/PennFudanPed/PNGImages/*.png    (PennFudanPed: the RGB frames, NOT the PedMasks)
#     neg_data/CamVid/*.png                    (CamVid 701_StillsRaw frames)

USE_NEG = any(f.lower().endswith((".png", ".jpg", ".jpeg"))
              for _dp, _ds, _fs in os.walk(NEG) for f in _fs)
print("negatives:", "ON (neg_data/ found)" if USE_NEG else "OFF (positives-only; no neg_data/)")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
def set_det(on: bool):
    torch.backends.cudnn.benchmark = (not on)
    torch.backends.cudnn.deterministic = on
    torch.use_deterministic_algorithms(on)
    torch.backends.cuda.matmul.allow_tf32 = (not on)
    torch.backends.cudnn.allow_tf32 = (not on)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

D = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", D)

def _collate(x):  # picklable identity collate (keeps PIL images / per-sample dicts)
    return x

In [ ]:
# === Download the released 3D-Mirage positives from HuggingFace ===
# Public dataset -> no token needed. For a private mirror, set HF_TOKEN in the env.
from huggingface_hub import snapshot_download
def _has_imgs(d):
    return os.path.isdir(d) and any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in os.listdir(d))

if not _has_imgs(ROOT):
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=ROOT,
                      allow_patterns=["[0-9]*.jpg", "[0-9]*.jpeg", "[0-9]*.png", "[0-9]*.json"],
                      token=os.environ.get("HF_TOKEN"))

n_img  = sum(1 for f in os.listdir(ROOT) if f.lower().endswith((".jpg", ".jpeg", ".png")))
n_json = sum(1 for f in os.listdir(ROOT) if f.lower().endswith(".json"))
print(f"{ROOT}: {n_img} images, {n_json} json")
assert n_img > 0 and n_json > 0, "download failed or empty ROOT"


In [ ]:
# === Dataset + sampler ===
def _wif(wid):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

def _sid(key, slot, seed):
    h = hashlib.sha1((str(seed) + "|" + str(slot) + "|" + str(key)).encode("utf-8")).digest()
    return int.from_bytes(h[:8], "big", signed=False)

def _jitter_color(im, p=0.2, g=None):
    g = random if g is None else g
    if g.random() > p:
        return im
    a = np.asarray(im).astype(np.float32)
    b = 1.0 + g.uniform(-0.05, 0.05)
    c = 1.0 + g.uniform(-0.05, 0.05)
    gg = 1.0
    a = a * b
    a = (a - 127.5) * c + 127.5
    a = np.clip(a, 1, 255)
    a = ((a / 255.0)**gg) * 255.0
    return Image.fromarray(a.clip(0, 255).astype(np.uint8))

def _hf_rois(im, rois, bbx):
    W, H = im.size
    im = im.transpose(Image.FLIP_LEFT_RIGHT)
    r2 = [r.copy() for r in rois]
    for r in r2:
        r[:, 0] = W - 1 - r[:, 0]
    x1, y1, x2, y2 = bbx
    bbx = (W - x2, y1, W - x1, y2)
    return im, r2, bbx

def _cov(b, rois):
    x1, y1, x2, y2 = b
    A = max(0.0, (x2 - x1) * (y2 - y1))
    if A <= 0:
        return 0.0
    R = Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)])
    G = None
    for r in rois:
        if r is None or len(r) < 3:
            continue
        try:
            P = Polygon(r).intersection(R)
            if P.is_empty:
                continue
            G = P if G is None else G.symmetric_difference(P)
        except:
            pass
    return (0.0 if (G is None) else (max(0.0, G.area) / A))

def _rects(W, H, rois, k=4, mc=0.40, tries=400):
    B = []
    for _ in range(tries):
        w = max(2, int(W * random.uniform(0.35, 0.95)))
        h = max(2, int(H * random.uniform(0.35, 0.95)))
        x1 = random.randint(0, max(0, W - w))
        y1 = random.randint(0, max(0, H - h))
        x2 = x1 + w
        y2 = y1 + h
        if _cov((x1, y1, x2, y2), rois) >= mc:
            B.append((x1, y1, x2, y2))
        if len(B) >= k:
            break
    if len(B) < k:
        xs = []
        ys = []
        for r in rois:
            if r is None or len(r) < 1:
                continue
            xs.extend(list(r[:, 0]))
            ys.extend(list(r[:, 1]))
        rx1, ry1, rx2, ry2 = (min(xs, default=0), min(ys, default=0), max(xs, default=W - 1), max(ys, default=H - 1))
        rcx, rcy = (0.5 * (rx1 + rx2), 0.5 * (ry1 + ry2))
        for _ in range(8 * k * 5):
            sf = random.uniform(1.0, 1.7)
            ww = max(2, int((rx2 - rx1) * sf))
            hh = max(2, int((ry2 - ry1) * sf))
            cx = rcx + random.uniform(-0.15, 0.15) * W
            cy = rcy + random.uniform(-0.15, 0.15) * H
            x1 = int(round(cx - 0.5 * ww))
            y1 = int(round(cy - 0.5 * hh))
            x2 = x1 + ww
            y2 = y1 + hh
            x1 = max(0, min(x1, W - 2))
            y1 = max(0, min(y1, H - 2))
            x2 = max(x1 + 2, min(x2, W))
            y2 = max(y1 + 2, min(y2, H))
            if _cov((x1, y1, x2, y2), rois) >= mc:
                B.append((x1, y1, x2, y2))
            if len(B) >= k:
                break
    if len(B) < k:
        for _ in range(2000):
            ww = max(2, int(W * random.uniform(0.5, 0.9)))
            hh = max(2, int(H * random.uniform(0.5, 0.9)))
            x1 = random.randint(0, max(0, W - ww))
            y1 = random.randint(0, max(0, H - hh))
            x2 = x1 + ww
            y2 = y1 + hh
            if _cov((x1, y1, x2, y2), rois) >= mc:
                B.append((x1, y1, x2, y2))
            if len(B) >= k:
                break
    if len(B) == 0:
        xs = []
        ys = []
        for r in rois:
            if r is None or len(r) < 1:
                continue
            xs.extend(list(r[:, 0]))
            ys.extend(list(r[:, 1]))
        rx1, ry1, rx2, ry2 = (min(xs, default=0), min(ys, default=0), max(xs, default=W - 1), max(ys, default=H - 1))
        rx1 = max(0, min(rx1, W - 2))
        ry1 = max(0, min(ry1, H - 2))
        rx2 = max(rx1 + 2, min(rx2, W))
        ry2 = max(ry1 + 2, min(ry2, H))
        B.append((rx1, ry1, rx2, ry2))
    while len(B) < k:
        B.append(B[len(B) % max(1, len(B))])
    return B[:k]

class DSP(Dataset):
    def __init__(self, root=ROOT, names=None, aug=True, rep=1, per=4, mc=0.40, seed=1337):
        random.seed(seed)
        np.random.seed(seed)
        self.s = []
        self.aug = aug
        self.rep = max(1, int(rep))
        self.per = int(per)
        self.mc = float(mc)
        self.seed = int(seed)
        g = random.Random(seed)
        for dp, ds, fs in os.walk(root):
            ds.sort()
            fs.sort()
            for f in fs:
                if not f.lower().endswith((".png", ".jpg", ".jpeg")):
                    continue
                n = os.path.splitext(f)[0]
                ip = os.path.join(dp, f)
                rj = os.path.join(dp, f"{n}.json")
                if not os.path.exists(rj):
                    continue
                key = os.path.join(os.path.relpath(dp, root), n)
                if (names is not None) and (key not in names):
                    continue
                try:
                    J = json.load(open(rj))
                except:
                    continue
                rois = [np.array(s.get("points", []), np.float32) for s in (J.get("shapes") or []) if isinstance(s, dict) and (s.get("label", "") == "illu") and (len(s.get("points", [])) >= 3)]
                if not rois:
                    continue
                try:
                    im = Image.open(ip).convert("RGB")
                    W, H = im.size
                    im.close()
                except:
                    continue
                bb = _rects(W, H, rois, k=self.per, mc=self.mc, tries=400)
                for b in bb:
                    self.s.append((ip, [r.copy() for r in rois], tuple(b), n))
        g.shuffle(self.s)
    def __len__(self):
        return len(self.s) * self.rep
    def __getitem__(self, i):
        j = i % len(self.s)
        slot = i // len(self.s)
        ip, rois, bbx, n = self.s[j]
        im = Image.open(ip).convert("RGB")
        W, H = im.size
        g = random.Random(_sid(str(ip) + "|" + str(bbx), slot, self.seed))
        if self.aug and g.random() < 0.5:
            im, rois, bbx = _hf_rois(im, rois, bbx)
        if self.aug:
            im = _jitter_color(im, p=0.2, g=g)
        cr = im.crop(bbx).resize(SZ, Image.Resampling.LANCZOS)
        return {"full": im, "crop": cr, "bbx": bbx, "rois": rois, "name": n}

def _rbbx(W, H, g):
    w = max(2, int(W * g.uniform(0.4, 0.9)))
    h = max(2, int(H * g.uniform(0.4, 0.9)))
    x1 = g.randint(0, max(0, W - w))
    y1 = g.randint(0, max(0, H - h))
    return (x1, y1, x1 + w, y1 + h)

class DSN(Dataset):
    def __init__(self, root=NEG, aug=True, rep=1, names=None, seed=1337):
        self.fs = []
        self.aug = aug
        self.rep = max(1, int(rep))
        self.seed = int(seed)
        for dp, ds, fs in os.walk(root):
            ds.sort()
            fs.sort()
            for f in fs:
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    p = os.path.join(dp, f)
                    if (names is not None) and (p not in names):
                        continue
                    self.fs.append(p)
    def __len__(self):
        return len(self.fs) * self.rep
    def __getitem__(self, i):
        j = i % len(self.fs)
        slot = i // len(self.fs)
        p = self.fs[j]
        im = Image.open(p).convert("RGB")
        W, H = im.size
        g = random.Random(_sid(p, slot, self.seed))
        if self.aug:
            if g.random() < 0.5:
                im = im.transpose(Image.FLIP_LEFT_RIGHT)
            im = _jitter_color(im, p=0.2, g=g)
            bbx = _rbbx(W, H, g)
        else:
            bbx = _rbbx(W, H, g)
        cr = im.crop(bbx).resize(SZ, Image.Resampling.LANCZOS)
        return {"full": im, "crop": cr, "bbx": bbx, "rois": [], "name": os.path.splitext(os.path.basename(p))[0], "neg": True}

def split_names(root=ROOT, val_ratio=0.10, test_ratio=0.10, seed=1337):
    names = []
    for dp, ds, fs in os.walk(root):
        ds.sort()
        fs.sort()
        for f in fs:
            if not f.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            n = os.path.splitext(f)[0]
            rj = os.path.join(dp, f"{n}.json")
            if not os.path.exists(rj):
                continue
            try:
                J = json.load(open(rj))
                ok = any((isinstance(s, dict) and s.get("label", "") == "illu" and len(s.get("points", [])) >= 3) for s in (J.get("shapes") or []))
            except:
                ok = False
            if ok:
                names.append(os.path.join(os.path.relpath(dp, root), n))
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(names))
    N = len(names)
    Nt = int(round(N * test_ratio))
    Nv = int(round(N * val_ratio))
    te = set([names[i] for i in idx[:Nt]])
    va = set([names[i] for i in idx[Nt:Nt + Nv]])
    tr = set([names[i] for i in idx[Nt + Nv:]])
    return tr, va, te

def split_neg(root=NEG, test_ratio=0.10, seed=1337):
    xs = []
    for dp, ds, fs in os.walk(root):
        ds.sort()
        fs.sort()
        for f in fs:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                xs.append(os.path.join(dp, f))
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(xs))
    N = len(xs)
    Nt = int(round(N * test_ratio))
    te = set(xs[i] for i in idx[:Nt])
    tr = set(xs[i] for i in idx[Nt:])
    return tr, te

class BBS(torch.utils.data.Sampler):
    def __init__(self, concat, batch_size=2, ratios=(1, 1), seed=1337, steps=None):
        self.ds = concat
        self.bs = int(batch_size)
        self.r = [float(x) for x in ratios]
        self.g = random.Random(seed)
        ofs = [0]
        for d in self.ds.datasets:
            ofs.append(ofs[-1] + len(d))
        self.idx = [list(range(ofs[i], ofs[i + 1])) for i in range(len(self.ds.datasets))]
        self.steps = steps if steps is not None else max(1, (sum(len(x) for x in self.idx) // self.bs))
    def __len__(self):
        return self.steps
    def __iter__(self):
        pools = [self.idx[i][:] for i in range(len(self.idx))]
        [self.g.shuffle(p) for p in pools]
        ptr = [0] * len(pools)
        def take(i):
            if not pools[i]:
                return None
            nonlocal ptr
            if ptr[i] >= len(pools[i]):
                self.g.shuffle(pools[i])
                ptr[i] = 0
            j = pools[i][ptr[i]]
            ptr[i] += 1
            return j
        k = len(pools)
        tot = max(1.0, sum(self.r))
        cum = [0] * k
        for t in range(1, self.steps + 1):
            tgt = [t * self.bs * (self.r[i] / tot) for i in range(k)]
            need = [tgt[i] - cum[i] for i in range(k)]
            order = list(range(k))
            order.sort(key=lambda i: need[i], reverse=True)
            b = []
            for _ in range(self.bs):
                j = order[0]
                x = take(j)
                if x is None:
                    j2 = max(range(k), key=lambda i: len(pools[i]))
                    x = take(j2)
                    j = j2
                b.append(x)
                cum[j] += 1
                need[j] -= 1
                order.sort(key=lambda i: need[i], reverse=True)
            self.g.shuffle(b)
            yield b

In [ ]:
# === Train/test split (seed 1337, 90/10) ===
set_det(True)
TR, _, TE = split_names(ROOT, val_ratio=0.0, test_ratio=TEST_RATIO, seed=SEED)
ALL = TR | TE
assert len(ALL) == 468, "expected 468 illu scenes, got %d -- incomplete download?" % len(ALL)
print("split: train=%d test=%d (of %d illu scenes)" % (len(TR), len(TE), len(ALL)))

os.makedirs("splits", exist_ok=True)
open("splits/train_ids.txt", "w").write("\n".join(sorted(os.path.normpath(x) for x in TR)))
open("splits/test_ids.txt",  "w").write("\n".join(sorted(os.path.normpath(x) for x in TE)))

ds_pos_tr = DSP(ROOT, names=TR, aug=True,  rep=AUG_MULT, per=4, mc=0.40, seed=SEED)
ds_pos_te = DSP(ROOT, names=TE, aug=False, rep=1,        per=4, mc=0.40, seed=SEED)

if USE_NEG:
    NR, NE = split_neg(NEG, test_ratio=TEST_RATIO, seed=SEED)
    ds_tr = ConcatDataset([ds_pos_tr, DSN(NEG, aug=True,  rep=AUG_MULT, names=NR, seed=SEED)])
    ds_te = ConcatDataset([ds_pos_te, DSN(NEG, aug=False, rep=1,        names=NE, seed=SEED)])
    _samp = BBS(ds_tr, batch_size=BS, ratios=(4, 1), seed=SEED)
    dl_tr = DataLoader(ds_tr, batch_sampler=_samp, num_workers=NW,
                       collate_fn=_collate, worker_init_fn=_wif, pin_memory=(D.type == "cuda"))
else:
    ds_tr, ds_te = ds_pos_tr, ds_pos_te
    _gen = torch.Generator(device="cpu"); _gen.manual_seed(SEED)
    dl_tr = DataLoader(ds_tr, batch_size=BS, shuffle=True, num_workers=NW, generator=_gen,
                       collate_fn=_collate, worker_init_fn=_wif, pin_memory=(D.type == "cuda"))

dl_te = DataLoader(ds_te, batch_size=BS, shuffle=False, num_workers=NW,
                   collate_fn=_collate, worker_init_fn=_wif, pin_memory=(D.type == "cuda"))
print("batches: train", len(dl_tr), "| test", len(dl_te),
      "|| samples: train", len(ds_tr), "| test", len(ds_te))

In [ ]:
# === Depth-normalize / Laplacian / ROI-mask helpers ===
def qn(x, m, lo=0.01, hi=0.99, eps=1e-6):
    o = []
    B = x.size(0)
    for i in range(B):
        xi = x[i]
        mi = (m[i] > 0)
        if mi.sum() < 32:
            o.append(torch.zeros_like(xi))
            continue
        v = xi[mi].float()
        a = torch.quantile(v, lo)
        b = torch.quantile(v, hi)
        y = ((xi.float() - a) / (b - a + eps)).clamp(0, 1).to(xi.dtype)
        o.append(y)
    return torch.stack(o, 0)

def t1(x):
    if x.dim() == 3:
        x = x.unsqueeze(1)
    if x.size(1) > 1:
        x = x[:, :1]
    return x.contiguous()

_KX = None
_KY = None
def L(x):
    global _KX, _KY
    if (_KX is None) or (_KX.device != x.device) or (_KX.dtype != x.dtype):
        _KX = torch.tensor([[1., -2., 1.]], device=x.device, dtype=x.dtype).view(1, 1, 1, 3)
        _KY = torch.tensor([[1.], [-2.], [1.]], device=x.device, dtype=x.dtype).view(1, 1, 3, 1)
    x2 = F.conv2d(F.pad(x, (1, 1, 0, 0), mode="replicate"), _KX)
    y2 = F.conv2d(F.pad(x, (0, 0, 1, 1), mode="replicate"), _KY)
    return torch.sqrt(x2 * x2 + y2 * y2 + 1e-12)

def imask(fs, bx, rois, ohw, dev):
    W, H = fs
    x1, y1, x2, y2 = bx
    Ho, Wo = ohw
    sx, sy = Wo / max(x2 - x1, 1e-6), Ho / max(y2 - y1, 1e-6)
    m = np.zeros((Ho, Wo), np.uint8)
    E = []
    R = []
    for P in rois:
        e = np.array((list(P.exterior.coords) if hasattr(P, "exterior") else P), np.float32)
        if e.ndim != 2 or e.shape[0] < 3:
            continue
        E.append(e.copy())
        try:
            R.append(Polygon(e))
        except:
            R.append(None)
    Hidx = set()
    for j in range(len(R)):
        if R[j] is None:
            continue
        for i in range(len(R)):
            if i == j or (R[i] is None):
                continue
            try:
                if R[i].buffer(2.0).covers(R[j]):
                    Hidx.add(j)
            except:
                pass
    for j, e in enumerate(E):
        ee = e.copy()
        ee[:, 0] = (ee[:, 0] - x1) * sx
        ee[:, 1] = (ee[:, 1] - y1) * sy
        t = np.zeros_like(m)
        cv2.fillPoly(t, [ee.astype(np.int32)], 1)
        m = np.bitwise_or(m, t)
        if R[j] is not None:
            for h in getattr(R[j], "interiors", []):
                hh = np.array(h.coords, np.float32)
                hh[:, 0] = (hh[:, 0] - x1) * sx
                hh[:, 1] = (hh[:, 1] - y1) * sy
                cv2.fillPoly(m, [hh.astype(np.int32)], 0)
    for j in Hidx:
        e = E[j].copy()
        e[:, 0] = (e[:, 0] - x1) * sx
        e[:, 1] = (e[:, 1] - y1) * sy
        cv2.fillPoly(m, [e.astype(np.int32)], 0)
    return torch.from_numpy(m).to(dev).view(1, 1, Ho, Wo).float()

def imask_full(fs, rois, ohw, dev):
    W, H = fs
    Ho, Wo = ohw
    sx, sy = Wo / max(W, 1e-6), Ho / max(H, 1e-6)
    m = np.zeros((Ho, Wo), np.uint8)
    E = []
    R = []
    for P in rois:
        e = np.array((list(P.exterior.coords) if hasattr(P, "exterior") else P), np.float32)
        if e.ndim != 2 or e.shape[0] < 3:
            continue
        E.append(e.copy())
        try:
            R.append(Polygon(e))
        except:
            R.append(None)
    Hidx = set()
    for j in range(len(R)):
        if R[j] is None:
            continue
        for i in range(len(R)):
            if i == j or (R[i] is None):
                continue
            try:
                if R[i].buffer(2.0).covers(R[j]):
                    Hidx.add(j)
            except:
                pass
    for j, e in enumerate(E):
        ee = e.copy()
        ee[:, 0] = ee[:, 0] * sx
        ee[:, 1] = ee[:, 1] * sy
        t = np.zeros_like(m)
        cv2.fillPoly(t, [ee.astype(np.int32)], 1)
        m = np.bitwise_or(m, t)
        if R[j] is not None:
            for h in getattr(R[j], "interiors", []):
                hh = np.array(h.coords, np.float32)
                hh[:, 0] = hh[:, 0] * sx
                hh[:, 1] = hh[:, 1] * sy
                cv2.fillPoly(m, [hh.astype(np.int32)], 0)
    for j in Hidx:
        e = E[j].copy()
        e[:, 0] = e[:, 0] * sx
        e[:, 1] = e[:, 1] * sy
        cv2.fillPoly(m, [e.astype(np.int32)], 0)
    return torch.from_numpy(m).to(dev).view(1, 1, Ho, Wo).float()

def _zn_stats(x, m, eps=1e-6):
    """Background z-normalization stats (mean/std under mask m), shared by the crop & full branches."""
    B = x.size(0)
    v = x.view(B, -1)
    w = m.view(B, -1)
    s = w.sum(1).clamp_min(1.0)
    mu = ((v * w).sum(1) / s).view(B, 1, 1, 1)
    sd = ((((v - mu.view(B, 1))**2) * w).sum(1) / s).clamp_min(eps).sqrt().view(B, 1, 1, 1)
    return mu, sd

# --- plane-fit + gate-feature helpers (full GSD: gated plane-mixture HKR) ---
def fit_plane(X, Y, Z, lam=1e-4):
    """Ridge-regularized least-squares plane z = a*X + b*Y + c over masked ROI points."""
    if Z.numel() < 16:
        return Z.new_tensor([0.0, 0.0, float(Z.mean() if Z.numel() > 0 else 0.0)])
    A = torch.stack([X, Y, torch.ones_like(X)], 1)
    AtA = A.T @ A + lam * torch.eye(3, device=A.device, dtype=A.dtype)
    Atb = A.T @ Z
    return torch.linalg.solve(AtA, Atb)

def plane_eval(a, b, c, X, Y):
    return a * X + b * Y + c

def roi_gate_feats(rmask, mroi, zT, zS, ring_sig, K):
    """10+K feature vector for the gate: roi/ring area fractions, teacher mean/std in roi & ring,
    seam magnitude, sigma min/max/mean, and the K plane-fit residuals."""
    B, _, H, W = zS.shape
    dev = zS.device
    dtype = zS.dtype
    mi = mroi[0, 0]
    ri = rmask[0, 0]
    nroi = float(mi.sum().item()) or 1.0
    nring = float(ri.sum().item()) or 1.0
    zt = zT[0, 0]
    zs = zS[0, 0]
    m_mu = float((zt * mi).sum().item() / nroi)
    m_sd = float((((zt - m_mu)**2) * mi).sum().item() / nroi + 1e-6)**0.5
    r_mu = float((zt * ri).sum().item() / nring)
    r_sd = float((((zt - r_mu)**2) * ri).sum().item() / nring + 1e-6)**0.5
    seam = float((ri * (zs - zt).abs()).mean().item())
    if torch.is_tensor(ring_sig):
        sigs = [float(x) for x in ring_sig.flatten().detach().tolist()]
    elif isinstance(ring_sig, (list, tuple)):
        sigs = [(float(x.item()) if torch.is_tensor(x) else float(x)) for x in ring_sig]
    else:
        sigs = [float(ring_sig)]
    if len(sigs) == 0:
        sigs = [1.0]
    s_min = min([x if np.isfinite(x) else 1.0 for x in sigs])
    s_max = max([x if np.isfinite(x) else 1.0 for x in sigs])
    s_mean = float(np.nanmean([x if np.isfinite(x) else 1.0 for x in sigs]))
    if len(sigs) < K:
        sigs = sigs + ([sigs[-1] if np.isfinite(sigs[-1]) else 1.0] * (K - len(sigs)))
    sigs = sigs[:K]
    sigs = [(x if np.isfinite(x) else 1.0) for x in sigs]
    f = [nroi/(H*W), nring/(H*W), m_mu, m_sd, r_mu, r_sd, seam, s_min, s_max, s_mean] + sigs
    return torch.tensor(f, device=dev, dtype=dtype).view(1, -1)

_MESH = {}
def _xy(z):
    """Cached normalized meshgrid X,Y in [-1,1] matching z's spatial size/device/dtype."""
    H, W = z.shape[-2], z.shape[-1]
    k = (H, W, z.device.type, str(z.dtype))
    if k in _MESH:
        return _MESH[k]
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, H, device=z.device, dtype=z.dtype),
                            torch.linspace(-1, 1, W, device=z.device, dtype=z.dtype), indexing='ij')
    X = xx.view(1, 1, H, W)
    Y = yy.view(1, 1, H, W)
    _MESH[k] = (X, Y)
    return X, Y

In [ ]:
!pip install -q --upgrade torchao peft

In [ ]:
# === Model: DAv2-Large student (LoRA r16) + frozen teacher ===
proc = AutoImageProcessor.from_pretrained(MID)
S = AutoModelForDepthEstimation.from_pretrained(MID).to(D)
T = AutoModelForDepthEstimation.from_pretrained(MID).to(D).eval()
T.requires_grad_(False)

BA = None
for c in ("backbone", "dpt", "vit"):
    if hasattr(S, c):
        BA = c
        break
assert BA is not None, "could not locate backbone attribute"
bb = getattr(S, BA)
md = dict(bb.named_modules())
pL = ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2", "q_proj", "k_proj", "v_proj", "o_proj"]
pC = ["embeddings.patch_embeddings.projection", "patch_embed.proj"]
isl = lambda m: (m.__class__.__name__ in ("Linear", "Conv1D"))
is2 = lambda m: (m.__class__.__name__ == "Conv2d")
tL = [n for n, m in md.items() if any(p in n for p in pL) and isl(m)]
tC = [n for n, m in md.items() if any(p in n for p in pC) and is2(m)]
if len(tL) == 0:
    tL = pL
if len(tC) == 0:
    for n, m in md.items():
        if is2(m):
            ks = getattr(m, "kernel_size", None)
            st = getattr(m, "stride", None)
            if ks and st and ks == st and max(ks) >= 8:
                tC.append(n)
tm = tL + tC if len(tC) > 0 else tL
lc = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                target_modules=tm, task_type="FEATURE_EXTRACTION")
bb = get_peft_model(bb, lc)
setattr(S, BA, bb)
for n, p in S.named_parameters():
    p.requires_grad_(False)
for n, p in getattr(S, BA).named_parameters():
    if "lora_" in n:
        p.requires_grad_(True)
n_train = sum(p.numel() for p in S.parameters() if p.requires_grad)
assert n_train > 0, "no trainable LoRA params"
print(f"LoRA trainable params: {n_train:,} | target modules: {len(tm)}")

K = 3
GateDim = 10 + K
GH = nn.Sequential(nn.Linear(GateDim, 64), nn.GELU(), nn.Linear(64, K + 1)).to(D)

opt = torch.optim.AdamW([p for p in S.parameters() if p.requires_grad] + list(GH.parameters()),
                        lr=LR, weight_decay=WD)

In [ ]:
# === Training: full GSD losses (crop + full)
set_det(False)
torch.backends.cudnn.benchmark = True
OUT = "./3dmirage_lora_best"
rad, radg = 3, 1

def _full_in(im):
    return im.resize(SZ, Image.Resampling.LANCZOS)

def _gate_roi_loop(rois_i, mfull_bool, mask_fn, zS, zT, X, Y, ring_fb):
    """Per-ROI gated plane-mixture HKR on one branch.
    Returns (Lroi_exp, LgateCE, Lent, Lroi_anchor_sum, nroi); Lroi_anchor_sum is a raw sum
    (caller averages over the batch total). ring_fb is the branch ring used when a per-ROI ring is tiny."""
    dev = zS.device
    Lroi_exp = zS.new_tensor(0.)
    LgateCE = zS.new_tensor(0.)
    Lent = zS.new_tensor(0.)
    Lroi_anchor = zS.new_tensor(0.)
    nroi = 0

    for P, is_neg in rois_i:
        mi = (mask_fn([P]) > 0) & mfull_bool
        mi = mi.float()
        if mi.sum() < 16:
            continue

        ri = (F.max_pool2d(mi, 2 * rad + 1, 1, rad) > 0).float() - mi
        ri = ri.clamp(0, 1)
        if int(ri.sum().item()) < 16:
            ri = (ring_fb > 0).float()

        mask2 = (ri[0, 0] > 0)
        if int(mask2.sum().item()) < 16:
            continue

        Xi = X[0, 0][mask2]
        Yi = Y[0, 0][mask2]
        Zi = zT[0, 0][mask2]

        planes = []
        sigs = []

        p0 = fit_plane(Xi, Yi, Zi, lam=1e-4)
        planes.append(p0)

        res0 = (
            (Zi - (p0[0] * Xi + p0[1] * Yi + p0[2])).abs().mean()
            if Zi.numel() > 0 else Zi.new_tensor(1.0)
        )
        sigs.append(res0)

        if Zi.numel() >= 128:
            for _ in range(2):
                idx = torch.randperm(Zi.numel(), device=dev)[:max(64, int(Zi.numel() * 0.5))]
                pk = fit_plane(Xi[idx], Yi[idx], Zi[idx], lam=1e-4)
                planes.append(pk)
                rk = (Zi - (pk[0] * Xi + pk[1] * Yi + pk[2])).abs().mean()
                sigs.append(rk)

        while len(planes) < K:
            planes.append(p0)
            sigs.append(res0)

        planes = planes[:K]
        sigs = sigs[:K]

        roi_den = mi.sum().clamp_min(1.)

        per_k = []
        for k in range(K):
            ak, bk, ck = planes[k][0], planes[k][1], planes[k][2]
            zp = plane_eval(ak, bk, ck, X, Y)
            per_k.append(((mi * (zS - zp).abs()).sum() / roi_den).view(()))

        per_k = torch.nan_to_num(torch.stack(per_k, 0), 0., 0., 0.)
        Lnull = torch.nan_to_num(((mi * (zS - zT).abs()).sum() / roi_den).view(()), 0., 0., 0.)

        sig = torch.stack(sigs).detach()
        sig = torch.nan_to_num(sig, 1., 1., 1.).clamp_min(1e-3)

        feats = torch.nan_to_num(roi_gate_feats(ri, mi, zT, zS, ring_sig=sig, K=K), 0., 0., 0.)
        logits = torch.nan_to_num(GH(feats), 0., 0., 0.)

        if not is_neg:
            logits[:, -1] = logits[:, -1] - 1e9

        w = F.softmax(logits * 1.5, dim=1).view(-1)
        w = torch.nan_to_num(w, 0., 0., 0.)
        w = w / (w.sum().clamp_min(1e-6))

        Lexp = torch.nan_to_num((w[:K] * per_k).sum() + w[K] * Lnull, 0., 0., 0.)
        Lroi_exp = Lroi_exp + Lexp
        Lroi_anchor = Lroi_anchor + per_k.min()
        nroi += 1

        tau, eps = 0.1, 0.05
        if is_neg:
            qq = torch.zeros(K + 1, device=w.device, dtype=w.dtype)
            qq[-1] = 1.0
            qq = qq * (1.0 - eps) + eps / float(K + 1)
        else:
            score = torch.exp((-sig / tau).clamp(-60, 60))
            score = score / score.sum().clamp_min(1e-6)
            qq = torch.cat([score, score.new_tensor([0.0])], 0)
            qq[:-1] = qq[:-1] * (1.0 - eps) + eps / float(K)
            qq[-1] = 0.0

        ce = torch.nan_to_num(-(qq * F.log_softmax(logits.view(-1), dim=0)).sum(), 0., 0., 0.)
        LgateCE = LgateCE + ce

        h = torch.nan_to_num(-(w * torch.log(w + 1e-9)).sum(), 0., 0., 0.)
        Lent = Lent + h

    return Lroi_exp, LgateCE, Lent, Lroi_anchor, nroi

def _seam_smooth(rf, zT, sG, kw, kh):
    """Separable-Gaussian ring-smoothed teacher on the low-gradient seam ring rf."""
    n1 = F.conv2d(F.pad(rf * zT, (sG // 2, sG // 2, 0, 0), mode="replicate"), kw)
    n2 = F.conv2d(F.pad(n1, (0, 0, sG // 2, sG // 2), mode="replicate"), kh)
    d1 = F.conv2d(F.pad(rf, (sG // 2, sG // 2, 0, 0), mode="replicate"), kw)
    d2 = F.conv2d(F.pad(d1, (0, 0, sG // 2, sG // 2), mode="replicate"), kh)
    return n2 / (d2 + 1e-6)

S.train()
GH.train()

best = float("inf")
t0 = time.time()
gstep = 0

TS = len(dl_tr) * max(1, E)
warm = max(0, int(0.10 * TS))  # warmup before checkpoint selection

sG = max(3, 4 * rad + 1)
_tt = torch.arange(sG, device=D, dtype=torch.float32) - (sG // 2)
_gk = torch.exp(-0.5 * (_tt / float(rad + 1.0)) ** 2)
_gk = _gk / (_gk.sum() + 1e-12)
kw = _gk.view(1, 1, 1, sG)
kh = _gk.view(1, 1, sG, 1)

for ep in range(E):
    for it, b in enumerate(dl_tr):
        ims = [u["full"] for u in b]
        crs = [u["crop"] for u in b]
        bbx = [u["bbx"] for u in b]
        rois = [(u["rois"] if isinstance(u.get("rois", []), list) else []) for u in b]
        negs = [bool(u.get("neg", False)) for u in b]

        si = proc(images=crs, return_tensors="pt").to(D)
        fi = proc(images=[_full_in(im) for im in ims], return_tensors="pt").to(D)

        # ---------- CROP branch ----------
        sp = t1(S(**si).predicted_depth)
        sp = torch.nan_to_num(sp, 0., 0., 0.)

        B, _, Hh, Wh = sp.shape

        with torch.no_grad():
            tc = t1(T(**si).predicted_depth).to(sp.dtype)
            tc = torch.nan_to_num(tc, 0., 0., 0.)

        m = torch.cat([
            imask(ims[i].size, bbx[i], rois[i], (Hh, Wh), sp.device)
            for i in range(B)
        ], 0)

        m_dil = (F.max_pool2d(m, 2 * rad + 1, 1, rad) > 0).float()
        ring_all = (m_dil - m).clamp(0, 1)

        m_dil2 = (F.max_pool2d(m_dil, 2 * radg + 1, 1, radg) > 0).float()
        ring_guard = (m_dil2 - m_dil).clamp(0, 1)

        bg = ((1. - m) * (1. - ring_all) * (1. - ring_guard))
        mbg = (bg > 0).float()
        mroi = (m > 0).float()

        has_bg = (bg.view(B, -1).sum(1) > 32).float().view(B, 1, 1, 1)
        mbg = mbg * has_bg

        muB, sB = _zn_stats(tc, mbg)
        zS = (sp - muB) / sB
        zT = (tc - muB) / sB

        Lf = (mroi * L(zS).abs()).sum() / (mroi.sum().clamp_min(1.))
        LbD = (mbg * (zS - zT).abs()).sum() / (mbg.sum().clamp_min(1.))
        LbL = (mbg * (L(zS) - L(zT)).abs()).sum() / (mbg.sum().clamp_min(1.))

        tLap = torch.abs(L(zT))
        ms = (ring_all > 0)
        q = (torch.quantile(tLap[ms], 0.90) if bool(ms.any()) else tLap.new_tensor(0.0))

        re = (ring_all * (tLap >= q).float())
        rf = (ring_all - re).clamp(0, 1)

        CbZ = _seam_smooth(rf, zT, sG, kw, kh)

        Lseam = (rf * (zS - CbZ).abs()).sum() / (rf.sum().clamp_min(1.))
        Ledge = (re * (L(zS) - L(zT)).abs()).sum() / (re.sum().clamp_min(1.))
        LedgeG = (ring_guard * (L(zS) - L(zT)).abs()).sum() / (ring_guard.sum().clamp_min(1.))

        X0, Y0 = _xy(sp)
        X = X0.expand_as(zS)
        Y = Y0.expand_as(zS)

        rl = [[(P, negs[i]) for P in rois[i]] for i in range(B)]

        Lroi_exp = sp.new_tensor(0.)
        LgateCE = sp.new_tensor(0.)
        Lent = sp.new_tensor(0.)
        Lroi_anchor = sp.new_tensor(0.)
        nroi2 = 0

        for i in range(B):
            mfb = (m[i:i + 1] > 0)
            le, lc_, lh, la, ni = _gate_roi_loop(
                rl[i],
                mfb,
                (lambda PP, ii=i: imask(ims[ii].size, bbx[ii], PP, (Hh, Wh), sp.device)),
                zS[i:i + 1],
                zT[i:i + 1],
                X[i:i + 1],
                Y[i:i + 1],
                ring_all[i:i + 1],
            )
            Lroi_exp = Lroi_exp + le
            LgateCE = LgateCE + lc_
            Lent = Lent + lh
            Lroi_anchor = Lroi_anchor + la
            nroi2 += ni

        Lroi_anchor = Lroi_anchor / float(max(1, nroi2))

        # ---------- FULL-IMAGE branch ----------
        sF = t1(S(**fi).predicted_depth)
        sF = torch.nan_to_num(sF, 0., 0., 0.)

        with torch.no_grad():
            TF = t1(T(**fi).predicted_depth).to(sF.dtype)
            TF = torch.nan_to_num(TF, 0., 0., 0.)

        Ho, Wo = sF.shape[-2:]

        mF = torch.cat([
            imask_full(ims[i].size, rois[i], (Ho, Wo), sF.device)
            for i in range(B)
        ], 0)

        m_dilF = (F.max_pool2d(mF, 2 * rad + 1, 1, rad) > 0).float()
        ringF = (m_dilF - mF).clamp(0, 1)

        m_dil2F = (F.max_pool2d(m_dilF, 2 * radg + 1, 1, radg) > 0).float()
        ringGF = (m_dil2F - m_dilF).clamp(0, 1)

        bgF = ((1. - mF) * (1. - ringF) * (1. - ringGF))
        mbgF = (bgF > 0).float()
        mroiF = (mF > 0).float()

        has_bgF = (bgF.view(B, -1).sum(1) > 32).float().view(B, 1, 1, 1)
        mbgF = mbgF * has_bgF

        muBF, sBF = _zn_stats(TF, mbgF)
        zSF = (sF - muBF) / sBF
        zTF = (TF - muBF) / sBF

        LbDF = (mbgF * (zSF - zTF).abs()).sum() / (mbgF.sum().clamp_min(1.))
        LbLF = (mbgF * (L(zSF) - L(zTF)).abs()).sum() / (mbgF.sum().clamp_min(1.))

        LfF = (mroiF * L(sF).abs()).sum() / (mroiF.sum().clamp_min(1.))

        tLapF = torch.abs(L(zTF))
        msF = (ringF > 0)
        qF = (torch.quantile(tLapF[msF], 0.90) if bool(msF.any()) else tLapF.new_tensor(0.0))

        reF = (ringF * (tLapF >= qF).float())
        rfF = (ringF - reF).clamp(0, 1)

        CbZF = _seam_smooth(rfF, zTF, sG, kw, kh)

        LseamF = (rfF * (zSF - CbZF).abs()).sum() / (rfF.sum().clamp_min(1.))
        LedgeF = (reF * (L(zSF) - L(zTF)).abs()).sum() / (reF.sum().clamp_min(1.))
        LedgeGF = (ringGF * (L(zSF) - L(zTF)).abs()).sum() / (ringGF.sum().clamp_min(1.))

        X0F, Y0F = _xy(sF)
        XF = X0F.expand_as(zSF)
        YF = Y0F.expand_as(zSF)

        Lroi_expF = sF.new_tensor(0.)
        LgateCEF = sF.new_tensor(0.)
        LentF = sF.new_tensor(0.)
        Lroi_anchorF = sF.new_tensor(0.)
        nroiF = 0

        for i in range(B):
            mfb = (mF[i:i + 1] > 0)
            le, lc_, lh, la, ni = _gate_roi_loop(
                rl[i],
                mfb,
                (lambda PP, ii=i: imask_full(ims[ii].size, PP, (Ho, Wo), sF.device)),
                zSF[i:i + 1],
                zTF[i:i + 1],
                XF[i:i + 1],
                YF[i:i + 1],
                ringF[i:i + 1],
            )
            Lroi_expF = Lroi_expF + le
            LgateCEF = LgateCEF + lc_
            LentF = LentF + lh
            Lroi_anchorF = Lroi_anchorF + la
            nroiF += ni

        Lroi_anchorF = Lroi_anchorF / float(max(1, nroiF))

        loss = (
            W_FLAT * Lf
            + W_BGD * LbD
            + W_BGL * LbL
            + W_SEAM * Lseam
            + W_EDGE * Ledge
            + W_EDGEG * LedgeG
            + W_ROI * Lroi_exp
            + W_GCE * LgateCE
            + W_GENT * Lent
            + W_ANCH * Lroi_anchor
            + W_FULL * (
                W_FLAT * LfF
                + W_BGD * LbDF
                + W_BGL * LbLF
                + W_SEAM * LseamF
                + W_EDGE * LedgeF
                + W_EDGEG * LedgeGF
                + W_ROI * Lroi_expF
                + W_GCE * LgateCEF
                + W_GENT * LentF
                + W_ANCH * Lroi_anchorF
            )
        )

        crit = float(loss.detach())
        if gstep >= warm and crit < best - 1e-9:
            best = crit
            os.makedirs(OUT, exist_ok=True)
            getattr(S, BA).save_pretrained(OUT)
            torch.save(GH.state_dict(), os.path.join(OUT, "gate.pt"))
            json.dump(
                {
                    "crit": best,
                    "ep": ep,
                    "it": it,
                    "g": gstep,
                    "time_s": time.time() - t0,
                },
                open(os.path.join(OUT, "meta.json"), "w"),
            )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(S.parameters()) + list(GH.parameters()), GRAD_CLIP)
        opt.step()

        if gstep % 20 == 0:
            print(
                f"ep{ep} it{it} g{gstep} | "
                f"Lf {float(Lf):.4f} "
                f"LbD {float(LbD):.4f} "
                f"LbL {float(LbL):.4f} "
                f"Lseam {float(Lseam):.4f} "
                f"Ledge {float(Ledge):.4f} "
                f"Lroi {float(Lroi_exp):.4f} "
                f"GCE {float(LgateCE):.4f} "
                f"anc {float(Lroi_anchor):.4f} | "
                f"loss {float(loss):.4f} "
                f"best {best:.4f}"
            )

        gstep += 1

print("training done | ckpt:", OUT, "(LoRA adapter + gate.pt)")

In [ ]:
# === Save the trained LoRA adapter (optional zip / download) ===
import datetime
assert os.path.isdir(OUT), f"missing best ckpt: {OUT} (did training run?)"
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP = f"./3dmirage_lora_{ts}.zip"
shutil.make_archive(ZIP[:-4], "zip", OUT)
print("saved adapter ->", OUT, "| zip ->", ZIP, f"({os.path.getsize(ZIP)/1e6:.2f} MB)")
try:
    from google.colab import files; files.download(ZIP)
except Exception as e:
    print("colab download skipped:", e)


In [ ]:
# === DCS / CCS evaluation on the held-out test split ===
import math, torch, torch.nn.functional as F, numpy as np, cv2
from PIL import Image as _Image

set_det(True)
S.eval()
T.eval()

@torch.no_grad()
def _pred_b(model, imgs):
    return t1(model(**proc(images=imgs, return_tensors="pt").to(D)).predicted_depth).float()

def _crop_from_full(dm, fs, bx, ohw):
    B, _, H, W = dm.shape
    x1, y1, x2, y2 = bx
    W0, H0 = fs
    Ho, Wo = ohw
    lx = x1 * (W / float(W0))
    rx = x2 * (W / float(W0))
    ty = y1 * (H / float(H0))
    by = y2 * (H / float(H0))
    xs = torch.linspace(lx, rx, Wo, device=dm.device, dtype=dm.dtype)
    ys = torch.linspace(ty, by, Ho, device=dm.device, dtype=dm.dtype)
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    gx = xx / (W - 1) * 2 - 1
    gy = yy / (H - 1) * 2 - 1
    grid = torch.stack([gx, gy], -1).view(1, Ho, Wo, 2).repeat(B, 1, 1, 1)
    return F.grid_sample(dm, grid, mode='bilinear', padding_mode='border', align_corners=True)

def _lap_stats(depth, mask):
    d = qn(depth, mask)
    l = L(d)
    v = l[mask > 0].view(-1)
    if v.numel() == 0:
        return 0.0, 0.0, 0.0
    k = max(1, int(math.ceil(0.10 * v.numel())))
    s_top = float(torch.topk(v, k, largest=True).values.sum().item())
    q = float(torch.quantile(v, 0.10).item())
    sel = v[v >= q]
    m10 = float(sel.mean().item()) if sel.numel() > 0 else 0.0
    a = (l[0, 0] * (mask[0, 0] > 0)).detach().cpu().numpy()
    vv = a[a > 0]
    if vv.size < 9:
        se = 0.0
    else:
        t = np.quantile(vv, 0.95)
        bw = (a > t).astype(np.uint8)
        num, lab = cv2.connectedComponents(bw, connectivity=8)
        if num <= 1:
            se = 0.0
        else:
            areas = sorted([int((lab == i).sum()) for i in range(1, num)], reverse=True)
            take = max(1, int(math.ceil(0.10 * len(areas))))
            se = float(sum(areas[:take])) / float(int((mask[0, 0] > 0).sum().item()))
    return s_top, m10, se

def _agg_dcs(p):
    if not p:
        return 0.0
    xs = np.array([x for x, _ in p], dtype=np.float64)
    ys = np.array([y for _, y in p], dtype=np.float64)
    return float(math.hypot(xs.mean(), ys.mean()) + np.mean(np.sqrt(xs * xs + ys * ys)))

def _agg_ccs(p):
    if not p:
        return 0.0
    xs = np.array([x for x, _ in p], dtype=np.float64)
    ys = np.array([y for _, y in p], dtype=np.float64)
    return float(abs(xs.mean() - ys.mean()) / math.sqrt(2.0) + np.mean(np.abs(xs - ys)) / math.sqrt(2.0))

def _agg_dcs_components(p):
    if not p:
        return 0.0, 0.0
    xs = np.array([x for x, _ in p], dtype=np.float64)
    ys = np.array([y for _, y in p], dtype=np.float64)
    return float(math.hypot(xs.mean(), ys.mean())), float(np.mean(np.sqrt(xs * xs + ys * ys)))

def _agg_ccs_components(p):
    if not p:
        return 0.0, 0.0
    xs = np.array([x for x, _ in p], dtype=np.float64)
    ys = np.array([y for _, y in p], dtype=np.float64)
    return float(abs(xs.mean() - ys.mean()) / math.sqrt(2.0)), float(np.mean(np.abs(xs - ys)) / math.sqrt(2.0))

# ---- eval loop (illusion-only; no negatives) ----
pairs_dcs_S = []
pairs_ccs_S = []
pairs_dcs_T = []
pairs_ccs_T = []
tot = 0
skipped = 0

print(f"[eval] DCS/CCS on dl_te ({len(dl_te)} batches)...")
with torch.no_grad():
    for bi, b in enumerate(dl_te):
        ims  = [u["full"] for u in b]
        crs  = [u["crop"] for u in b]
        bbx  = [u["bbx"]  for u in b]
        rois = [(u["rois"] if isinstance(u.get("rois", []), list) else []) for u in b]
        negs = [bool(u.get("neg", False)) for u in b]

        sc = _pred_b(S, crs)
        tc = _pred_b(T, crs)
        Sf = _pred_b(S, [im.resize(SZ, _Image.Resampling.LANCZOS) for im in ims])
        Tf = _pred_b(T, [im.resize(SZ, _Image.Resampling.LANCZOS) for im in ims])
        Bn, _, Hh, Wh = sc.shape

        for i in range(Bn):
            if negs[i] or not rois[i]:
                skipped += 1
                continue
            m = imask(ims[i].size, bbx[i], rois[i], (Hh, Wh), sc.device)
            if int(m.sum().item()) < 16:
                skipped += 1
                continue

            Td = _crop_from_full(Tf[i:i+1], ims[i].size, bbx[i], (Hh, Wh))
            Sd = _crop_from_full(Sf[i:i+1], ims[i].size, bbx[i], (Hh, Wh))

            dcs_x_S, ccs_x_S, _ = _lap_stats(Sd,        m)
            dcs_y_S, ccs_y_S, _ = _lap_stats(sc[i:i+1], m)
            dcs_x_T, ccs_x_T, _ = _lap_stats(Td,        m)
            dcs_y_T, ccs_y_T, _ = _lap_stats(tc[i:i+1], m)

            pairs_dcs_S.append((dcs_x_S, dcs_y_S))
            pairs_ccs_S.append((ccs_x_S, ccs_y_S))
            pairs_dcs_T.append((dcs_x_T, dcs_y_T))
            pairs_ccs_T.append((ccs_x_T, ccs_y_T))
            tot += 1
        if (bi + 1) % 10 == 0:
            print(f"  batch {bi+1}/{len(dl_te)} | samples: {tot}")

DCS_S = _agg_dcs(pairs_dcs_S)
CCS_S = _agg_ccs(pairs_ccs_S)
DCS_T = _agg_dcs(pairs_dcs_T)
CCS_T = _agg_ccs(pairs_ccs_T)
DCS_S_dc, DCS_S_da = _agg_dcs_components(pairs_dcs_S)
DCS_T_dc, DCS_T_da = _agg_dcs_components(pairs_dcs_T)
CCS_S_dc, CCS_S_da = _agg_ccs_components(pairs_ccs_S)
CCS_T_dc, CCS_T_da = _agg_ccs_components(pairs_ccs_T)

print()
print("=" * 90)
print(f"  DAv2-L DCS/CCS over {tot} illusion ROIs")
print("=" * 90)
print(f"  {'':20s}  {'dcluster':>10s} {'davg':>10s} {'DCS':>10s}    {'Dcluster':>12s} {'Davg':>12s} {'CCS':>12s}")
print(f"  {'Teacher (baseline)':20s}  {DCS_T_dc:10.2f} {DCS_T_da:10.2f} {DCS_T:10.2f}    {CCS_T_dc:12.4e} {CCS_T_da:12.4e} {CCS_T:12.4e}")
print(f"  {'Student (LoRA)':20s}  {DCS_S_dc:10.2f} {DCS_S_da:10.2f} {DCS_S:10.2f}    {CCS_S_dc:12.4e} {CCS_S_da:12.4e} {CCS_S:12.4e}")
if DCS_T > 0:
    pct_dc  = (DCS_S_dc-DCS_T_dc)/DCS_T_dc*100 if DCS_T_dc else 0.0
    pct_da  = (DCS_S_da-DCS_T_da)/DCS_T_da*100 if DCS_T_da else 0.0
    pct_dcs = (DCS_S-DCS_T)/DCS_T*100
    pct_ccs = (CCS_S-CCS_T)/CCS_T*100 if CCS_T else 0.0
    print(f"  {'Delta % (S vs T)':20s}  {pct_dc:+10.2f} {pct_da:+10.2f} {pct_dcs:+10.2f}    {'':12s} {'':12s} {pct_ccs:+12.2f}")
print("=" * 90)

In [ ]:
# === Background preservation) evaluation
import torch, torch.nn.functional as F, numpy as np
from PIL import Image as _Image

set_det(True)
torch.manual_seed(42)
S.eval()
T.eval()

@torch.no_grad()
def _pred_b(model, imgs):
    return t1(model(**proc(images=imgs, return_tensors="pt").to(D)).predicted_depth).float()

def _crop_from_full(dm, fs, bx, ohw):
    B, _, H, W = dm.shape
    x1, y1, x2, y2 = bx
    W0, H0 = fs
    Ho, Wo = ohw
    lx = x1 * (W / float(W0))
    rx = x2 * (W / float(W0))
    ty = y1 * (H / float(H0))
    by = y2 * (H / float(H0))
    xs = torch.linspace(lx, rx, Wo, device=dm.device, dtype=dm.dtype)
    ys = torch.linspace(ty, by, Ho, device=dm.device, dtype=dm.dtype)
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    gx = xx / (W - 1) * 2 - 1
    gy = yy / (H - 1) * 2 - 1
    grid = torch.stack([gx, gy], -1).view(1, Ho, Wo, 2).repeat(B, 1, 1, 1)
    return F.grid_sample(dm, grid, mode='bilinear', padding_mode='border', align_corners=True)

def _ss_fit(s, t, m, eps=1e-6):
    sv = torch.nan_to_num(s[m > 0].view(-1).float(), 0.0, 0.0, 0.0)
    tv = torch.nan_to_num(t[m > 0].view(-1).float(), 0.0, 0.0, 0.0)
    if sv.numel() < 32:
        return 1.0, 0.0
    ms = sv.mean()
    mt = tv.mean()
    vs = ((sv - ms)**2).mean()
    cst = ((sv - ms)*(tv - mt)).mean()
    a = float((cst / (vs + eps)).clamp_min(0.0).item())
    b = float((mt - a * ms).item())
    return a, b

_r2_list = []
_r_list = []
_mae_iqr_list = []
tot = 0
skipped = 0

print(f"[R2] running on dl_te ({len(dl_te)} batches)...")
with torch.no_grad():
    for bi, b in enumerate(dl_te):
        ims  = [u["full"] for u in b]
        crs  = [u["crop"] for u in b]
        bbx  = [u["bbx"]  for u in b]
        rois = [(u["rois"] if isinstance(u.get("rois", []), list) else []) for u in b]
        negs = [bool(u.get("neg", False)) for u in b]

        sc = _pred_b(S, crs)
        Tf = _pred_b(T, [im.resize(SZ, _Image.Resampling.LANCZOS) for im in ims])
        Bn, _, Hh, Wh = sc.shape

        for i in range(Bn):
            if negs[i] or not rois[i]:
                skipped += 1
                continue
            m = imask(ims[i].size, bbx[i], rois[i], (Hh, Wh), sc.device)
            if int(m.sum().item()) < 16:
                skipped += 1
                continue

            rmax = 4
            bg = (((1.0 - m) * (1.0 - F.max_pool2d(m, 2*rmax+1, 1, rmax))) > 0.5).float()
            if int(bg.sum().item()) < 32:
                skipped += 1
                continue

            Td = _crop_from_full(Tf[i:i+1], ims[i].size, bbx[i], (Hh, Wh))
            raw_sp = sc[i:i+1]

            a, b_coeff = _ss_fit(raw_sp, Td, bg)
            spa = a * raw_sp + b_coeff

            mb = (bg > 0)
            tb = Td[mb].view(-1)
            sb = spa[mb].view(-1)

            mt = tb.mean()
            sst = ((tb - mt)**2).sum().clamp_min(1e-9)
            sse = ((sb - tb)**2).sum()
            _r2_list.append(float((1.0 - sse / sst).item()))

            sbm = sb.mean()
            rnum = ((tb - mt) * (sb - sbm)).mean()
            rden = tb.std() * sb.std() + 1e-9
            _r_list.append(float((rnum / rden).item()))

            q25, q75 = torch.quantile(tb, torch.tensor([0.25, 0.75], device=tb.device))
            iqr = (q75 - q25).clamp_min(1e-9)
            mae = (sb - tb).abs().mean()
            _mae_iqr_list.append(float((100.0 * mae / iqr).item()))
            tot += 1
        if (bi + 1) % 10 == 0:
            print(f"  batch {bi+1}/{len(dl_te)} | samples: {tot}")

R2_mean  = float(np.mean(_r2_list))      if _r2_list      else float('nan')
r_mean   = float(np.mean(_r_list))       if _r_list       else float('nan')
mae_iqr  = float(np.mean(_mae_iqr_list)) if _mae_iqr_list else float('nan')

print()
print("=" * 50)
print(f"  Background Preservation (crop-view) - {tot} samples")
print("=" * 50)
print(f"  BG R^2 (aligned):     {R2_mean*100:.2f}%")
print(f"  Pearson r (bg):       {r_mean:.4f}")
print(f"  MAE/IQR (bg):         {mae_iqr:.2f}%")
print("=" * 50)